### 5. Neural networks
##### Explore varies neural networks using keras library.

In [5]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping



In [37]:
# first use the basic no text data
data = pd.read_csv("../COMP30027_2024_asst2_data/train_select.csv")
test_data = pd.read_csv("../COMP30027_2024_asst2_data/test_select.csv")
# drop id before calculation, add back after
data = data.drop(columns='id')
test_data = test_data.drop(columns='id')

In [47]:
# split the data
X = data.drop(columns=["imdb_score_binned"])
y = data["imdb_score_binned"]



X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=715)

# set up scalers
# mmscaler = MinMaxScaler()
ssscaler = StandardScaler()

ssX = ssscaler.fit_transform(X)
# mmscaler.fit(X)
# ssscaler.fit(X_train)

# mmX = mmscaler.transform(X)
ssX_train = ssscaler.fit_transform(X_train)
ssX_test = ssscaler.fit_transform(X_test)



In [50]:
# design nn
model = Sequential([
    Dense(8, input_shape=(X_train.shape[1],), activation='relu'),
    Dense(4, activation='sigmoid'),
    Dense(y.nunique(), activation='softmax')
])

# instantiate
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

history = model.fit(ssX_train, y_train, epochs=100, batch_size=8, validation_split=0.2, callbacks=[early_stopping])


# get accuracy
test_loss, test_acc = model.evaluate(ssX_test, y_test)
print("Test Accuracy:" ,test_acc)


Epoch 1/100
241/241 [==============================] - 1s 1ms/step - loss: 1.6337 - accuracy: 0.2539 - val_loss: 1.3914 - val_accuracy: 0.6258
Epoch 2/100
241/241 [==============================] - 0s 1ms/step - loss: 1.2569 - accuracy: 0.6082 - val_loss: 1.1219 - val_accuracy: 0.6279
Epoch 3/100
241/241 [==============================] - 0s 1ms/step - loss: 1.0728 - accuracy: 0.6082 - val_loss: 0.9941 - val_accuracy: 0.6279
Epoch 4/100
241/241 [==============================] - 0s 1ms/step - loss: 0.9837 - accuracy: 0.6087 - val_loss: 0.9304 - val_accuracy: 0.6299
Epoch 5/100
241/241 [==============================] - 0s 1ms/step - loss: 0.9346 - accuracy: 0.6228 - val_loss: 0.8958 - val_accuracy: 0.6528
Epoch 6/100
241/241 [==============================] - 0s 1ms/step - loss: 0.9039 - accuracy: 0.6405 - val_loss: 0.8743 - val_accuracy: 0.6798
Epoch 7/100
241/241 [==============================] - 0s 1ms/step - loss: 0.8824 - accuracy: 0.6613 - val_loss: 0.8577 - val_accuracy: 0.6861

#### Kaggle prediction on test data

In [52]:
sstest_data = ssscaler.fit_transform(test_data)

history = model.fit(ssX, y, epochs=100, batch_size=32, validation_split=0.2, callbacks=[early_stopping])

Epoch 1/100
76/76 [==============================] - 0s 2ms/step - loss: 0.6734 - accuracy: 0.7199 - val_loss: 0.6680 - val_accuracy: 0.7338
Epoch 2/100
76/76 [==============================] - 0s 2ms/step - loss: 0.6710 - accuracy: 0.7224 - val_loss: 0.6702 - val_accuracy: 0.7321
Epoch 3/100
76/76 [==============================] - 0s 1ms/step - loss: 0.6690 - accuracy: 0.7191 - val_loss: 0.6720 - val_accuracy: 0.7288
Epoch 4/100
76/76 [==============================] - 0s 1ms/step - loss: 0.6679 - accuracy: 0.7224 - val_loss: 0.6737 - val_accuracy: 0.7271
Epoch 5/100
76/76 [==============================] - 0s 1ms/step - loss: 0.6669 - accuracy: 0.7233 - val_loss: 0.6743 - val_accuracy: 0.7271
Epoch 6/100
76/76 [==============================] - 0s 2ms/step - loss: 0.6657 - accuracy: 0.7203 - val_loss: 0.6754 - val_accuracy: 0.7271
Epoch 7/100
76/76 [==============================] - 0s 1ms/step - loss: 0.6651 - accuracy: 0.7199 - val_loss: 0.6764 - val_accuracy: 0.7238
Epoch 8/100
7

In [56]:
test_pred = model.predict(test_data)
test_pred = test_pred.argmax(axis=1)
ids = range(1, len(test_pred) + 1)

# save submission file
prediction = pd.DataFrame({'id': ids, 'imdb_score_binned': test_pred})
print(prediction['imdb_score_binned'].value_counts())
prediction.to_csv("../Kaggle_submissions/dense_nn_select_feature.csv", index=False)

24/24 [==============================] - 0s 1ms/step
2    722
4     24
3      6
Name: imdb_score_binned, dtype: int64


### 5.2

In [14]:
data = pd.read_csv("../COMP30027_2024_asst2_data/train_select.csv")
# drop id before calculation, add back after
data = data.drop(columns='id')

In [15]:
# split the data
X = data.drop(columns=["imdb_score_binned"])
y = data["imdb_score_binned"]

# set up scalers
mmscaler = MinMaxScaler()
ssscaler = StandardScaler()

mmscaler.fit(X)
ssscaler.fit(X)

mmX = mmscaler.transform(X)
ssX = ssscaler.transform(X)

X_train, X_test, y_train, y_test = train_test_split(ssX, y, test_size=0.2, random_state=715)

In [16]:
# design nn
model = Sequential([
    Dense(32, input_shape=(X_train.shape[1],), activation='relu'),
    Dense(16, activation='relu'), 
    Dense(y.nunique(), activation='softmax')
])

# instantiate
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

history = model.fit(X_train, y_train, epochs=50, batch_size=32, validation_split=0.2)

# get accuracy
test_loss, test_acc = model.evaluate(X_test, y_test)  # Make sure to transform X_test with the same scaler used on X_train
print("Test Accuracy:" ,test_acc)


Epoch 1/50
61/61 [==============================] - 1s 3ms/step - loss: 1.4697 - accuracy: 0.3564 - val_loss: 1.2488 - val_accuracy: 0.6486
Epoch 2/50
61/61 [==============================] - 0s 2ms/step - loss: 1.1523 - accuracy: 0.6394 - val_loss: 1.0154 - val_accuracy: 0.6507
Epoch 3/50
61/61 [==============================] - 0s 2ms/step - loss: 0.9881 - accuracy: 0.6504 - val_loss: 0.8974 - val_accuracy: 0.6736
Epoch 4/50
61/61 [==============================] - 0s 2ms/step - loss: 0.8990 - accuracy: 0.6696 - val_loss: 0.8350 - val_accuracy: 0.6778
Epoch 5/50
61/61 [==============================] - 0s 2ms/step - loss: 0.8460 - accuracy: 0.6769 - val_loss: 0.8024 - val_accuracy: 0.6944
Epoch 6/50
61/61 [==============================] - 0s 1ms/step - loss: 0.8110 - accuracy: 0.6857 - val_loss: 0.7858 - val_accuracy: 0.6902
Epoch 7/50
61/61 [==============================] - 0s 2ms/step - loss: 0.7904 - accuracy: 0.6920 - val_loss: 0.7717 - val_accuracy: 0.6881
Epoch 8/50
61/61 [==

In [13]:
test_data = ssscaler.fit_transform(test_data)
test_pred = model.predict(test_data)
test_pred = test_pred.argmax(axis=1)
ids = range(1, len(test_pred) + 1)

print(test_pred.shape)
# save submission file
prediction = pd.DataFrame({'id': ids, 'imdb_score_binned': test_pred})
prediction.to_csv("../Kaggle_submissions/dense_nn_notext.csv", index=False)

24/24 [==============================] - 0s 1ms/step
(752,)
